In [0]:
from snapi_py_client.snapi_bridge import StocknoteAPIPythonBridge
samco=StocknoteAPIPythonBridge()

In [0]:
userId = dbutils.secrets.get(scope='scope-test-learning-awg', key='samco-username')
password = dbutils.secrets.get(scope='scope-test-learning-awg', key='samco-password')
yob = dbutils.secrets.get(scope='scope-test-learning-awg', key='samco-yob')
login=samco.login(body={"userId":userId,'password':password,'yob':yob})
print("Login details",login)

In [0]:
samco.set_session_token(sessionToken=eval(login)["sessionToken"])

In [0]:
print(samco.get_quote(symbol_name="SBIN", exchange=samco.EXCHANGE_NSE))

In [0]:
value=[{"symbol":"-53"}]
value_symbol = value[0]['symbol']
print(value_symbol)

In [0]:
from queue import Queue
import threading
import time
import json

# -------- DEFINE MESSAGE HANDLER --------
def handle_streaming_data(message):
    data = json.loads(message)
    print("Streaming message:", data)

tick_queue = Queue()
output_path = "abfss://samco-streaming-container@datalaketestlearningawg.dfs.core.windows.net/source"

stop_event = threading.Event()

def handle_streaming_data(message):
    data = json.loads(message)
    tick_queue.put(data)

def flush_worker(symboll):
    batch = []
    last_flush = time.time()
    while not stop_event.is_set():
        try:
            msg = tick_queue.get(timeout=1)
            batch.append(msg)
        except Exception as e:
            print("FLUSH WORKER QUEUE ERRORRRRRRRRRRRR : ", e)
        if time.time() - last_flush >= 5:
            if batch:
                file_name = f"{output_path}/{symboll}/ticks_{int(time.time())}.json"
                with open(file_name, "w") as f:
                    for msg in batch:
                        f.write(msg + "\n")
                print("Flushed", len(batch), "ticks")
                batch = []
            last_flush = time.time()

worker_thread = threading.Thread(target=flush_worker, args=(value_symbol,), daemon=True)
worker_thread.start()

def handle_on_close(x,y,z):
    print ("Connection Closed XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

# attach handler
samco.on_message = handle_streaming_data
samco.on_close = handle_on_close
samco.set_streaming_data(value)

# -------- START STREAM --------
samco.start_streaming()

In [0]:
worker_thread.stop()

In [0]:
dbutils.notebook.exit("stop")